# 14b — Quand le sur-ajustement au simulateur apparaît : priver le réseau de la volatilité

## Le résultat du 14, et la question qu'il pose

Au notebook 14, le réseau entraîné sur $P_0$ restait robuste hors modèle, et la randomisation n'apportait rien. On avait conclu : le sur-ajustement ne se produit que si la politique optimale dépend d'un paramètre **que le réseau n'observe pas**. Ici il observait la variance instantanée $v$, donc il s'adaptait au niveau de vol tout seul.

Mais **en pratique, $v$ n'est pas observable** : c'est un état caché du modèle. On observe des proxys (volatilité implicite, VIX, vol réalisée), bruités et décalés. La question honnête est donc : que se passe-t-il quand le réseau **ne voit pas** le niveau de vol ?

Ce notebook le teste en retirant $v$ des features. On s'attend alors à ce qu'un réseau entraîné sur $P_0$ (vol ~20%) devienne **fragile au scénario de vol à 30%** (S2), parce qu'il couvre comme si la vol était toujours 20%. Et on vérifie que la **randomisation** répare cette fragilité. C'est la démonstration contrôlée du mécanisme.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.stats import norm
from scipy.optimize import brentq

torch.manual_seed(0)
S0, K, mu, r, T = 100., 100., 0.05, 0.02, 1.0
n, cost, alpha = 63, 0.01, 0.95; dt = T/n

def heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T):
    out = []
    for u, b in [(0.5, kappa - rho*xi), (-0.5, kappa)]:
        d = np.sqrt((rho*xi*1j*phi - b)**2 - xi**2*(2*u*1j*phi - phi**2))
        g = (b - rho*xi*1j*phi + d)/(b - rho*xi*1j*phi - d)
        C = r*1j*phi*T + (kappa*theta/xi**2)*((b - rho*xi*1j*phi + d)*T - 2*np.log((1-g*np.exp(d*T))/(1-g)))
        D = (b - rho*xi*1j*phi + d)/xi**2 * ((1-np.exp(d*T))/(1-g*np.exp(d*T)))
        out.append(np.exp(C + D*v0 + 1j*phi*np.log(S0)))
    return out
def heston_call(S0, v0, r, kappa, theta, xi, rho, T, K):
    def integ(phi, i):
        f = heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T)[i]
        return (np.exp(-1j*phi*np.log(K))*f/(1j*phi)).real
    P1 = 0.5 + quad(integ, 1e-8, 200, args=(0,), limit=200)[0]/np.pi
    P2 = 0.5 + quad(integ, 1e-8, 200, args=(1,), limit=200)[0]/np.pi
    return S0*P1 - K*np.exp(-r*T)*P2
def bs_price(S,K,tau,r,s): d1=(np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)); return S*norm.cdf(d1)-K*np.exp(-r*tau)*norm.cdf(d1-s*np.sqrt(tau))
def bs_delta(S,K,tau,r,s): return norm.cdf((np.log(S/K)+(r+0.5*s**2)*tau)/(s*np.sqrt(tau)))
def cvar(p,a=0.95): l=-p; return l[l>=np.quantile(l,a)].mean()

P0 = (0.04, 2.0, 0.04, 0.3, -0.7)
scen = {'P0 in-model':                P0,
        'S1 xi=0.6':                  (0.04, 2.0, 0.04, 0.6, -0.7),
        'S2 vol 30%':                 (0.09, 2.0, 0.09, 0.3, -0.7),
        'S3 rho=-0.3':                (0.04, 2.0, 0.04, 0.3, -0.3),
        'S4 crash':                   (0.04, 2.0, 0.04, 0.5, -0.9)}
prem0 = heston_call(S0, P0[0], r, P0[1], P0[2], P0[3], P0[4], T, K)

def sim_np(par, m, seed):
    v0, kappa, theta, xi, rho = par; rng = np.random.default_rng(seed)
    S = np.empty((m, n+1)); v = np.empty((m, n+1)); S[:,0] = S0; v[:,0] = v0
    for k in range(n):
        Z1 = rng.standard_normal(m); Z2 = rho*Z1 + np.sqrt(1-rho**2)*rng.standard_normal(m)
        vk = np.maximum(v[:,k], 0.)
        v[:,k+1] = np.maximum(v[:,k] + kappa*(theta-vk)*dt + xi*np.sqrt(vk*dt)*Z2, 0.)
        S[:,k+1] = S[:,k]*np.exp((mu - 0.5*vk)*dt + np.sqrt(vk*dt)*Z1)
    return S, v
def band_hedge(S, premium, sig, band=0.13):
    m = S.shape[0]; times = np.linspace(0, T, n+1); cash = np.full(m, premium); pos = np.zeros(m)
    for k in range(n):
        tau = max(T-times[k], 1e-3); tgt = bs_delta(S[:,k], K, tau, r, sig)
        tr = np.where(np.abs(tgt-pos) > band, tgt-pos, 0.); cash -= tr*S[:,k] + cost*np.abs(tr)*S[:,k]; pos += tr; cash *= np.exp(r*dt)
    return cash + pos*S[:,-1] - np.maximum(S[:,-1]-K, 0.)

# reference classique (recalibree par scenario)
classic = {}
for name, par in scen.items():
    prem = heston_call(S0, par[0], r, par[1], par[2], par[3], par[4], T, K)
    sig = brentq(lambda s: bs_price(S0, K, T, r, s) - prem, 1e-3, 2.0)
    S, _ = sim_np(par, 60_000, 7)
    classic[name] = (prem, cvar(band_hedge(S, prem, sig)))
print("reference classique prete")


## Le réseau AVEUGLE à la volatilité

Seul changement par rapport au notebook 14 : l'état d'entrée n'a plus que **3 features**, $[\log(S/K), \tau, \text{position}]$. On retire $v$. Le réseau ne peut donc plus savoir dans quel régime de vol il se trouve.


In [ ]:
def heston_paths_t(par, m):
    v0, kappa, theta, xi, rho = par
    S = torch.full((m,), S0); v = torch.full((m,), float(v0)); Ss = [S]
    for k in range(n):
        Z1 = torch.randn(m); Z2 = rho*Z1 + np.sqrt(1-rho**2)*torch.randn(m)
        vk = torch.clamp(v, min=0.)
        v = torch.clamp(v + kappa*(theta-vk)*dt + xi*torch.sqrt(vk*dt)*Z2, min=0.)
        S = S*torch.exp((mu - 0.5*vk)*dt + torch.sqrt(vk*dt)*Z1)
        Ss.append(S)
    return torch.stack(Ss, 1)                      # on ne renvoie que S : v n'est pas observe

def cvar_torch(L, a=0.95):
    var = torch.quantile(L, a); return L[L >= var].mean()

class BlindNet(torch.nn.Module):
    def __init__(self, h=32):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(3, h), torch.nn.ReLU(),
                                       torch.nn.Linear(h, h), torch.nn.ReLU(), torch.nn.Linear(h, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

def hedge_blind(net, S, premium):
    m = S.shape[0]; cash = torch.full((m,), premium); pos = torch.zeros(m)
    for k in range(n):
        tau = float(T - k*dt)
        feat = torch.stack([torch.log(S[:,k]/K), torch.full((m,), tau), pos], dim=1)   # PAS de v
        d = net(feat); tr = d - pos
        cash = cash - tr*S[:,k] - cost*torch.abs(tr)*S[:,k]; cash = cash*np.exp(r*dt); pos = d
    return cash + pos*S[:,-1] - torch.clamp(S[:,-1]-K, min=0.0)

def train_blind(sampler, epochs=400, m=20_000, lr=1e-3):
    net = BlindNet(); opt = torch.optim.Adam(net.parameters(), lr=lr)
    for ep in range(epochs):
        S = heston_paths_t(sampler(), m)
        loss = cvar_torch(-hedge_blind(net, S.detach(), prem0))
        opt.zero_grad(); loss.backward(); opt.step()
    return net

def eval_blind(net, par, prem, m=60_000, seed=7):
    Snp, _ = sim_np(par, m, seed)
    with torch.no_grad():
        pnl = hedge_blind(net, torch.tensor(Snp, dtype=torch.float32), prem).numpy()
    return cvar(pnl)

print("entrainement aveugle sur P0 ..."); blind_p0 = train_blind(lambda: P0)


In [ ]:
rng_dr = np.random.default_rng(0)
def sampler_dr():
    theta = rng_dr.uniform(0.02, 0.09); xi = rng_dr.uniform(0.2, 0.6); rho = rng_dr.uniform(-0.9, -0.3)
    return (theta, 2.0, theta, xi, rho)
print("entrainement aveugle randomise ..."); blind_dr = train_blind(sampler_dr, epochs=600)

print(f"{'scenario':16s} {'classique':>10s} {'aveugle P0':>11s} {'aveugle DR':>11s}")
bP0, bDR = {}, {}
for name, par in scen.items():
    prem = classic[name][0]
    bP0[name] = eval_blind(blind_p0, par, prem); bDR[name] = eval_blind(blind_dr, par, prem)
    print(f"{name:16s} {classic[name][1]:10.2f} {bP0[name]:11.2f} {bDR[name]:11.2f}")


In [ ]:
labels = list(scen.keys()); x = np.arange(len(labels)); wd = 0.27
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x-wd, [classic[k][1] for k in labels], wd, label='classique (recalibre)', color='tab:orange')
ax.bar(x,    [bP0[k]        for k in labels], wd, label='aveugle a v, entraine P0', color='tab:red')
ax.bar(x+wd, [bDR[k]        for k in labels], wd, label='aveugle a v, randomise', color='tab:green')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('CVaR 95%'); ax.set_title("Reseau AVEUGLE au niveau de vol : fragile sur S2, repare par randomisation")
ax.legend(); fig.tight_layout(); plt.show()


## Lecture attendue

Le point de comparaison est **S2 (vol 30%)**. Le réseau aveugle entraîné sur $P_0$ a appris à couvrir pour une vol de 20% et ne peut pas savoir qu'elle est passée à 30%, donc il devrait sous-couvrir et voir son CVaR bondir, potentiellement au-dessus du classique (qui, lui, recalibre sa vol implicite). Le réseau aveugle **randomisé**, entraîné sur toute une plage de niveaux de vol, apprend une politique plus prudente et devrait rattraper une bonne partie de l'écart sur S2.

Contraste avec le notebook 14 : là, le réseau *voyait* $v$, donc pas de fragilité et randomisation inutile. Ici, aveugle, la fragilité apparaît sur la dimension cachée (le niveau de vol) et la randomisation devient utile. C'est la démonstration propre du principe : **le sur-ajustement au simulateur apparaît exactement sur les paramètres que le réseau n'observe pas.**


## Et en pratique ?

$v$ (la variance instantanée) n'est pas observable : c'est un état caché. En vrai, on donne au réseau un **proxy observable** du niveau de vol, typiquement la **volatilité implicite** cotée (ou le VIX pour un indice), voire la vol réalisée récente. Ces proxys tracent $v$ de près, donc en pratique on se situe entre les deux extrêmes de cette étude :

- réseau qui voit $v$ (notebook 14) : idéalisation, borne haute de robustesse ;
- réseau aveugle (ce notebook) : cas extrême sans signal de vol ;
- réalité : réseau nourri d'un proxy bruité, robuste au niveau de vol tant que le proxy est fiable, fragile si le proxy décroche du vrai régime.

La leçon opérationnelle : **la robustesse d'un deep hedger dépend directement de la qualité de son signal de vol**. Un desk investit donc autant dans le proxy de vol qu'il donne au réseau que dans le réseau lui-même. Et pour les paramètres qu'aucun proxy ne capture bien ($\xi$, $\rho$), la randomisation à l'entraînement reste l'assurance.
